In [0]:
# =============================================================
# CREDIT-RISK MODEL TOOL
#
# Requires : 10_runtime_config (run first via %run)
# Globals used: REGISTERED_MODEL_URI, MODEL_CONFIGURATION_FILE
# Exposes  : calculate_credit_risk(borrower_id) -> dict
# =============================================================

import json
from typing import Any

import mlflow
import mlflow.pyfunc
import pandas as pd

# REGISTERED_MODEL_URI and MODEL_CONFIGURATION_FILE are in scope
# from 10_runtime_config via %run.

# Demo cohort path is fixed — not in runtime_configuration.json.
DEMO_COHORT_FILE = (
    "/Volumes/demodatabrciks123/"
    "creditrisk/raw_file/structured/"
    "demo_cohort/demo_borrower_cohort.parquet"
)

# ---------------------------------------------------------
# Load frozen model configuration
# ---------------------------------------------------------
with open(MODEL_CONFIGURATION_FILE, "r", encoding="utf-8") as file:
    locked_model_configuration = json.load(file)

MODEL_FEATURE_COLUMNS   = locked_model_configuration["feature_columns"]
LOCKED_REVIEW_THRESHOLD = float(locked_model_configuration["locked_threshold"])

assert len(MODEL_FEATURE_COLUMNS) == 64

# ---------------------------------------------------------
# Load the governed model once per session
# ---------------------------------------------------------
mlflow.set_registry_uri("databricks-uc")
credit_risk_model = mlflow.pyfunc.load_model(REGISTERED_MODEL_URI)

# ---------------------------------------------------------
# Load the 30 demo borrowers
# ---------------------------------------------------------
demo_borrowers = pd.read_parquet(DEMO_COHORT_FILE)

assert demo_borrowers["borrower_id"].is_unique

missing_model_features = [
    col for col in MODEL_FEATURE_COLUMNS
    if col not in demo_borrowers.columns
]
assert not missing_model_features, (
    f"Missing model features: {missing_model_features}"
)

# ---------------------------------------------------------
# Scoring tool
# ---------------------------------------------------------
def calculate_credit_risk(
    borrower_id: str,
) -> dict[str, Any]:
    """
    Score one demonstration borrower using the governed,
    calibrated Unity Catalog model.

    Returns decision-support information only.
    Does not approve or reject a loan.
    """
    if not isinstance(borrower_id, str):
        raise TypeError("borrower_id must be a string.")

    borrower_id = borrower_id.strip().upper()

    if not borrower_id.startswith("B"):
        raise ValueError(
            "borrower_id must use the expected B-format, "
            "for example B000187."
        )

    borrower_rows = demo_borrowers[
        demo_borrowers["borrower_id"] == borrower_id
    ]

    if borrower_rows.empty:
        raise ValueError(
            f"Borrower {borrower_id} was not found "
            "in the 30-borrower demonstration cohort."
        )

    if len(borrower_rows) != 1:
        raise RuntimeError(f"Borrower {borrower_id} has multiple records.")

    model_input  = borrower_rows[MODEL_FEATURE_COLUMNS].copy()
    model_output = credit_risk_model.predict(model_input)

    if len(model_output) != 1:
        raise RuntimeError(
            "The risk model returned an unexpected number of rows."
        )

    prediction         = model_output.iloc[0]
    probability        = float(prediction["probability_of_bankruptcy"])
    returned_threshold = float(prediction["review_threshold"])
    review_required    = bool(prediction["review_required"])
    status             = str(prediction["decision_support_status"])

    if abs(returned_threshold - LOCKED_REVIEW_THRESHOLD) > 1e-12:
        raise RuntimeError(
            "The registered model threshold differs from "
            "the locked configuration."
        )

    if not 0.0 <= probability <= 1.0:
        raise RuntimeError("The model returned an invalid probability.")

    if review_required != (probability >= LOCKED_REVIEW_THRESHOLD):
        raise RuntimeError(
            "The returned review flag is inconsistent "
            "with the locked threshold."
        )

    return {
        "borrower_id":               borrower_id,
        "probability_of_bankruptcy": round(probability, 6),
        "review_threshold":          round(LOCKED_REVIEW_THRESHOLD, 6),
        "review_required":           review_required,
        "decision_support_status":   status,
        "model_name": (
            "demodatabrciks123.creditrisk."
            "credit_risk_assessment_model"
        ),
        "model_alias":   "Candidate",
        "model_version": "1",
        "purpose": (
            "Human decision support; not an automatic "
            "lending decision."
        ),
    }


print("calculate_credit_risk() defined.")
print(f"  Model URI : {REGISTERED_MODEL_URI}")
print(f"  Threshold : {LOCKED_REVIEW_THRESHOLD:.8f}")
print(f"  Borrowers : {len(demo_borrowers)} in cohort")